In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

# File paths (edit to yours)

clinical_file = "/content/drive/MyDrive/5703/All_Subjects_DXSUM_26Aug2025_clinical_patient_data.csv"
gene_file     = "/content/drive/MyDrive/5703/ADNI_Gene_Expression_Profile_transposed.csv"

df_clin = pd.read_csv(clinical_file, low_memory=False)
df_gene = pd.read_csv(gene_file,     low_memory=False)

#3) Normalize join key
if "PTID" in df_clin.columns and "subject_id" not in df_clin.columns:
    df_clin = df_clin.rename(columns={"PTID": "subject_id"})
if "SubjectID" in df_gene.columns and "subject_id" not in df_gene.columns:
    df_gene = df_gene.rename(columns={"SubjectID": "subject_id"})

df_clin["subject_id"] = df_clin["subject_id"].astype(str)
df_gene["subject_id"] = df_gene["subject_id"].astype(str)


# 4) Keep labeled rows (DIAGNOSIS not null)
dfc = df_clin[~df_clin["DIAGNOSIS"].isna()].copy()

def select_baseline(g):
    if "VISCODE" in g.columns:
        is_bl = g["VISCODE"].astype(str).str.lower() == "bl"
        if is_bl.any():
            return g[is_bl].head(1)
    if "EXAMDATE" in g.columns:
        gg = g.copy()
        gg["_dt"] = pd.to_datetime(gg["EXAMDATE"], errors="coerce")
        gg = gg.sort_values(by=["_dt","EXAMDATE"])
        return gg.head(1).drop(columns=["_dt"])
    return g.head(1)

dfc_one = dfc.groupby("subject_id", group_keys=False).apply(select_baseline)


# 5) Inner-join with gene data (gene-driven task)

gene_cols = [c for c in df_gene.columns if c != "subject_id"]
for c in gene_cols:
    df_gene[c] = pd.to_numeric(df_gene[c], errors="coerce")

merged = pd.merge(dfc_one, df_gene, on="subject_id", how="inner")

# 6) Light QC on gene features
X = merged[gene_cols]
missing_rate = X.isna().mean(axis=0)
X = X.loc[:, missing_rate < 0.50]
X = X.apply(lambda col: col.fillna(col.median()), axis=0)

meta_cols = [c for c in ["subject_id","VISCODE","EXAMDATE","PHASE","DIAGNOSIS"] if c in merged.columns]
df_full = pd.concat([merged[meta_cols].reset_index(drop=True),
                     X.reset_index(drop=True)], axis=1)

print("Full usable rows:", len(df_full), "| genes kept:", X.shape[1])

# 7) Save full dataset (label = DIAGNOSIS numeric)
out_full = "/content/drive/MyDrive/5703/ADNI_gene_DIAGNOSIS_numeric_full.csv"
df_full.to_csv(out_full, index=False)
print("Saved:", out_full)

# 8) Stratified 5k sample by DIAGNOSIS (numeric)
TARGET_N = 5000
if len(df_full) > TARGET_N and "DIAGNOSIS" in df_full.columns:
    df_5k = (
        df_full.groupby("DIAGNOSIS", group_keys=False)
               .apply(lambda g: g.sample(
                   n=max(1, int(round(TARGET_N * len(g)/len(df_full)))),
                   random_state=42))
               .reset_index(drop=True)
    )
    if len(df_5k) > TARGET_N:
        df_5k = df_5k.sample(n=TARGET_N, random_state=42).reset_index(drop=True)
else:
    df_5k = df_full.copy()

out_5k = "/content/drive/MyDrive/ADNI_gene_DIAGNOSIS_numeric_5000.csv"
df_5k.to_csv(out_5k, index=False)
print("Saved:", out_5k, "| rows:", len(df_5k))


In [ ]:
import pandas as pd
from sklearn.feature_selection import VarianceThreshold


df = pd.read_csv("/content/drive/MyDrive/5703/ADNI_gene_DIAGNOSIS_numeric_full.csv", low_memory=False)

meta_cols = ["subject_id","VISCODE","EXAMDATE","PHASE","DIAGNOSIS"]
meta = df[meta_cols]
X = df.drop(columns=meta_cols, errors="ignore")

selector = VarianceThreshold(threshold=0.01)
X_selected = selector.fit_transform(X)

selected_genes = X.columns[selector.get_support(indices=True)]
X_opt = pd.DataFrame(X_selected, columns=selected_genes)


df_opt = pd.concat([meta.reset_index(drop=True), X_opt.reset_index(drop=True)], axis=1)

out_file = "/content/drive/MyDrive/RNA_1.csv"
df_opt.to_csv(out_file, index=False)
print("✅ saved:", out_file)


In [ ]:
import pandas as pd
import numpy as np

# 1. Load data
file_path = "/content/drive/MyDrive/5703/RNA_1.csv"
df = pd.read_csv(file_path, low_memory=False)

print("Original shape:", df.shape)

# 2. Split numeric and non-numeric columns
non_numeric = df.select_dtypes(exclude=[np.number])
numeric = df.select_dtypes(include=[np.number])

print("Non-numeric cols:", non_numeric.shape[1],
      "Numeric cols:", numeric.shape[1])

# 3. Missing value handling
missing_ratio = numeric.isna().mean()
numeric = numeric.loc[:, missing_ratio <= 0.3]
numeric = numeric.fillna(numeric.median())

print("After missing value handling:", numeric.shape)

# 4. Feature selection
variances = numeric.var().sort_values(ascending=False)
top_genes = variances.head(20000).index
numeric = numeric[top_genes]

print("After variance filtering:", numeric.shape)

# 5. Merge back non-numeric columns
df_cleaned = pd.concat([non_numeric, numeric], axis=1)

print("Final shape:", df_cleaned.shape)

# 6. Save cleaned dataset
output_path = "/content/drive/MyDrive/5703/RNA_cleaned_20k.csv"
df_cleaned.to_csv(output_path, index=False)
print("Cleaned file saved to:", output_path)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
import gc

# 1. Load dataset
file_path = "/content/drive/MyDrive/5703/RNA_cleaned_20k.csv"
df = pd.read_csv(file_path, low_memory=False)

print("Original shape:", df.shape)

# 2. Separate metadata and features
meta_cols = ["DIAGNOSIS"]
non_numeric = df[meta_cols + list(df.select_dtypes(exclude=[np.number]).columns.unique())]

numeric = df.drop(columns=non_numeric.columns, errors="ignore")

print("Metadata cols:", non_numeric.shape[1],
      "Feature cols:", numeric.shape[1])

# 2. Convert all features to numeric
numeric = numeric.apply(pd.to_numeric, errors="coerce")

# Handle missing values with median
imputer = SimpleImputer(strategy="median")
numeric[:] = imputer.fit_transform(numeric)
print("After missing value handling:", numeric.shape)

gc.collect()

# 3. Handle outliers (IQR clipping)
Q1 = numeric.quantile(0.25)
Q3 = numeric.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Clip values outside the bounds
numeric = numeric.clip(lower=lower_bound, upper=upper_bound, axis=1)
print("After outlier handling:", numeric.shape)

gc.collect()

# 4. Standardization (Z-score)
scaler = StandardScaler()
numeric[:] = scaler.fit_transform(numeric)
print("After standardization:", numeric.shape)

# 5. Remove redundancy (correlation filter)
corr_matrix = numeric.corr().abs()

to_drop = set()
for i in range(len(corr_matrix.columns)):
    if corr_matrix.columns[i] in to_drop:
        continue
    high_corr = corr_matrix.index[(corr_matrix.iloc[i, :] > 0.95)].tolist()
    if corr_matrix.columns[i] in high_corr:
        high_corr.remove(corr_matrix.columns[i])
    to_drop.update(high_corr)

numeric = numeric.drop(columns=list(to_drop))
print("After redundancy removal:", numeric.shape)


# 7. Merge metadata back
df_final = pd.concat([non_numeric, numeric], axis=1)
print("Final dataset shape:", df_final.shape)

# 8. Save final dataset
output_path = "/content/drive/MyDrive/5703/RNA_final_processed.csv"
df_final.to_csv(output_path, index=False)
print("✅ Processed dataset saved to:", output_path)


In [ ]:
import pandas as pd

# 1. Load data
file_path = "/content/drive/MyDrive/5703/RNA_final_processed.csv"
df = pd.read_csv(file_path, low_memory=False)

print("Original shape:", df.shape)

# 2. Drop irrelevant columns
drop_cols = ["PTID", "RID", "VISCODE", "EXAMDATE", "PHASE", "STUDY", "COHORT", "SITE"]

df = df.drop(columns=[c for c in drop_cols if c in df.columns])

print("After dropping irrelevant columns:", df.shape)

# 3. Save new dataset
output_path = "/content/drive/MyDrive/5703/RNA_cleaned.csv"
df.to_csv(output_path, index=False)

print("✅ Filtered dataset saved to:", output_path)
